# preparing

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

# Locate the repo root by walking up until project.duckdb is found
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project.duckdb").exists())
SPLIT_DIR = ROOT / "data" / "processed"
SPLIT_NAMES = ["train_supervised", "train_pseudo_unlabeled", "validation", "test"]

# One emoji character (main emoji/symbol blocks plus variation selector)
EMOJI = "[\U0001F000-\U0001FAFF\u2300-\u23FF\u2600-\u27BF\u2B00-\u2BFF\uFE0F]"
# One word character (letter, digit, underscore, or ZWNJ)
WORD = r"[\w\u200c]"

# Zero-width boundaries: they match only the emoji itself
EMOJI_AFTER_WORD = rf"(?<={WORD}){EMOJI}"
EMOJI_BEFORE_WORD = rf"{EMOJI}(?={WORD})"
EMOJI_GLUED = rf"{EMOJI_AFTER_WORD}|{EMOJI_BEFORE_WORD}"

summary = []
for name in SPLIT_NAMES:
    df = pd.read_csv(SPLIT_DIR / f"{name}.csv", dtype=str)
    body = df["body"].fillna("")

    print(f"\n=== {name} ===")
    print("columns:", list(df.columns), "| rows:", len(df))

    has_after_word = body.str.contains(EMOJI_AFTER_WORD, regex=True)
    has_before_word = body.str.contains(EMOJI_BEFORE_WORD, regex=True)
    has_glued = has_after_word | has_before_word

    glued_body = body[has_glued]
    n_spaces = glued_body.str.count(EMOJI_AFTER_WORD).sum() + glued_body.str.count(EMOJI_BEFORE_WORD).sum()

    kinds = Counter()
    for text in glued_body:
        kinds.update(re.findall(EMOJI_GLUED, text))

    print(f"rows with word+emoji glued   : {has_after_word.sum():,}")
    print(f"rows with emoji+word glued   : {has_before_word.sum():,}")
    print(f"rows with either (to edit)   : {has_glued.sum():,} ({has_glued.mean() * 100:.2f}%)")
    print(f"total spaces to insert       : {n_spaces:,}")
    print("distinct emojis:", len(kinds))
    print("top 15 emojis  :", kinds.most_common(15))
    print("samples:")
    for text in glued_body.head(3):
        print("  ", text[:150])

    summary.append({
        "file": name,
        "rows": len(df),
        "rows_to_edit": int(has_glued.sum()),
        "spaces_to_insert": int(n_spaces),
    })

print()
print(pd.DataFrame(summary))


=== train_supervised ===
columns: ['id', 'body', 'product_id', 'recommendation_status', 'step1_rec_label', 'rating_band', 'step1_rate_band_label', 'rate'] | rows: 2860236
rows with word+emoji glued   : 25,377
rows with emoji+word glued   : 1
rows with either (to edit)   : 25,377 (0.89%)
total spaces to insert       : 26,463
distinct emojis: 458
top 15 emojis  : [('😍', 3506), ('👌', 3159), ('👍', 2170), ('❤', 1174), ('🙏', 1088), ('😂', 1015), ('😐', 728), ('😊', 661), ('🥰', 636), ('😁', 617), ('🌹', 474), ('😅', 461), ('😑', 373), ('🤩', 346), ('🥲', 338)]
samples:
   هنوز استفاده نکردم ولی طرحش قشنگه😍🔥 لازم نیست حتماا اندازه ناخن ببرید. میتونید طرح و قیچی کنید بزارید رو ناخن بعدش با استون و براش یا گوش پاک کن دورشو
   درود تراش سالم به دستم رسید و از کیفیت قابل قبولی هم برخورداره از تیغه ها راضی بودم فقط همونطور که در تصویر مشاهده میفرمایید من رنگ " بنفش " سفارش داد
   کیفیتش و سایزش خوبه ولی یجوریه انگار با گواش رنگش کردن🤦🏻♀️ احساس میکنی اگه با دست خیس بهش دست بزنی رنگش پاک میشه یا دستت رنگ میگ

# Cell 2: Modification on 4 files

In [2]:
import os
import re
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project.duckdb").exists())
SPLIT_DIR = ROOT / "data" / "processed"
SPLIT_NAMES = ["train_supervised", "train_pseudo_unlabeled", "validation", "test"]

EMOJI = "[\U0001F000-\U0001FAFF\u2300-\u23FF\u2600-\u27BF\u2B00-\u2BFF\uFE0F]"
WORD = r"[\w\u200c]"

# Same boundaries as the counting cell; the (?!\uFE0F) guard keeps keycap
# sequences such as digit + FE0F + 20E3 from being split
EMOJI_AFTER_WORD = rf"(?<={WORD})(?!\uFE0F){EMOJI}"
EMOJI_BEFORE_WORD = rf"{EMOJI}(?={WORD})"

# (rows_to_edit, spaces_to_insert) printed by the counting cell
EXPECTED = {
    "train_supervised": (25377, 26463),
    "train_pseudo_unlabeled": (5312, 5579),
    "validation": (6667, 6974),
    "test": (6385, 6705),
}

for name in SPLIT_NAMES:
    path = SPLIT_DIR / f"{name}.csv"
    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    old = df["body"]
    mask = old.str.contains(EMOJI_AFTER_WORD, regex=True) | old.str.contains(EMOJI_BEFORE_WORD, regex=True)

    new = (
        old[mask]
        .str.replace(EMOJI_AFTER_WORD, r" \g<0>", regex=True)
        .str.replace(EMOJI_BEFORE_WORD, r"\g<0> ", regex=True)
    )

    n_rows = int(mask.sum())
    n_spaces = int((new.str.len() - old[mask].str.len()).sum())

    df.loc[mask, "body"] = new

    # No glued emoji may remain, and no other row may have changed
    left = df["body"].str.contains(EMOJI_AFTER_WORD, regex=True) | df["body"].str.contains(EMOJI_BEFORE_WORD, regex=True)
    assert not left.any(), f"{name}: glued emojis remain"
    assert (old[~mask] == df.loc[~mask, "body"]).all(), f"{name}: untouched rows changed"

    tmp = path.with_suffix(".csv.tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    os.replace(tmp, path)

    status = "OK" if (n_rows, n_spaces) == EXPECTED[name] else "DIFFERENT FROM COUNTING CELL"
    print(f"{name}: rows edited {n_rows:,} | spaces inserted {n_spaces:,} | {status}")

train_supervised: rows edited 25,374 | spaces inserted 26,454 | DIFFERENT FROM COUNTING CELL
train_pseudo_unlabeled: rows edited 5,310 | spaces inserted 5,577 | DIFFERENT FROM COUNTING CELL
validation: rows edited 6,666 | spaces inserted 6,973 | DIFFERENT FROM COUNTING CELL
test: rows edited 6,385 | spaces inserted 6,705 | OK


# again Cell 1

In [3]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

# Locate the repo root by walking up until project.duckdb is found
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project.duckdb").exists())
SPLIT_DIR = ROOT / "data" / "processed"
SPLIT_NAMES = ["train_supervised", "train_pseudo_unlabeled", "validation", "test"]

# One emoji character (main emoji/symbol blocks plus variation selector)
EMOJI = "[\U0001F000-\U0001FAFF\u2300-\u23FF\u2600-\u27BF\u2B00-\u2BFF\uFE0F]"
# One word character (letter, digit, underscore, or ZWNJ)
WORD = r"[\w\u200c]"

# Zero-width boundaries: they match only the emoji itself
EMOJI_AFTER_WORD = rf"(?<={WORD}){EMOJI}"
EMOJI_BEFORE_WORD = rf"{EMOJI}(?={WORD})"
EMOJI_GLUED = rf"{EMOJI_AFTER_WORD}|{EMOJI_BEFORE_WORD}"

summary = []
for name in SPLIT_NAMES:
    df = pd.read_csv(SPLIT_DIR / f"{name}.csv", dtype=str)
    body = df["body"].fillna("")

    print(f"\n=== {name} ===")
    print("columns:", list(df.columns), "| rows:", len(df))

    has_after_word = body.str.contains(EMOJI_AFTER_WORD, regex=True)
    has_before_word = body.str.contains(EMOJI_BEFORE_WORD, regex=True)
    has_glued = has_after_word | has_before_word

    glued_body = body[has_glued]
    n_spaces = glued_body.str.count(EMOJI_AFTER_WORD).sum() + glued_body.str.count(EMOJI_BEFORE_WORD).sum()

    kinds = Counter()
    for text in glued_body:
        kinds.update(re.findall(EMOJI_GLUED, text))

    print(f"rows with word+emoji glued   : {has_after_word.sum():,}")
    print(f"rows with emoji+word glued   : {has_before_word.sum():,}")
    print(f"rows with either (to edit)   : {has_glued.sum():,} ({has_glued.mean() * 100:.2f}%)")
    print(f"total spaces to insert       : {n_spaces:,}")
    print("distinct emojis:", len(kinds))
    print("top 15 emojis  :", kinds.most_common(15))
    print("samples:")
    for text in glued_body.head(3):
        print("  ", text[:150])

    summary.append({
        "file": name,
        "rows": len(df),
        "rows_to_edit": int(has_glued.sum()),
        "spaces_to_insert": int(n_spaces),
    })

print()
print(pd.DataFrame(summary))


=== train_supervised ===
columns: ['id', 'body', 'product_id', 'recommendation_status', 'step1_rec_label', 'rating_band', 'step1_rate_band_label', 'rate'] | rows: 2860236
rows with word+emoji glued   : 4
rows with emoji+word glued   : 0
rows with either (to edit)   : 4 (0.00%)
total spaces to insert       : 9
distinct emojis: 1
top 15 emojis  : [('️', 9)]
samples:
   سری کتاب های ۴۰۰۰ essential english words واقعا خیلی عالین واسیه ار زبان آموزی چه self study چه با معلم خیلی مفیده میتونین فیلمو مشاهده بکنین نقد و بر
   سری کتاب های ۴۰۰۰ essential english words واقعا خیلی عالین واسیه ار زبان آموزی چه self study چه با معلم خیلی مفیده میتونین فیلمو مشاهده بکنین نقد و بر
   خیلی عالیی 💖💖👌👌👏👏 خیلی ناراحتم این مجموعه تموم شد ولی در عوض در این مدتی که این مجموعه رو میخوندم واقعا خیلی خیلی لذت می‌بردم و حتما بهتون پیشنهاد می‌

=== train_pseudo_unlabeled ===
columns: ['id', 'body'] | rows: 635824
rows with word+emoji glued   : 2
rows with emoji+word glued   : 0
rows with either (to edit)   : 2 

# Complementary Check and Controll with original `clean_base` table

In [4]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project.duckdb").exists())
SPLIT_DIR = ROOT / "data" / "processed"
SPLIT_NAMES = ["train_supervised", "train_pseudo_unlabeled", "validation", "test"]

con = duckdb.connect(str(ROOT / "project.duckdb"), read_only=True)

frames = []
for name in SPLIT_NAMES:
    csv_path = (SPLIT_DIR / f"{name}.csv").as_posix()
    frames.append(con.execute(f"""
        SELECT '{name}' AS split,
               c.recommendation_status AS class,
               count(*) AS rows,
               sum((s.body IS DISTINCT FROM c.body)::INT) AS edited
        FROM read_csv('{csv_path}', all_varchar=true) s
        JOIN clean_base c ON s.id = c.id::VARCHAR
        GROUP BY 1, 2
        ORDER BY 1, 2
    """).df())

result = pd.concat(frames, ignore_index=True)
result["pct_edited"] = (result["edited"] / result["rows"] * 100).round(2)
print(result)
print("total edited:", result["edited"].sum())

                     split            class     rows   edited  pct_edited
0         train_supervised          no_idea   292757   1637.0        0.56
1         train_supervised  not_recommended   266817   1772.0        0.66
2         train_supervised      recommended  1832583  17957.0        0.98
3         train_supervised              NaN   468079   4008.0        0.86
4   train_pseudo_unlabeled          no_idea    76428    454.0        0.59
5   train_pseudo_unlabeled  not_recommended    68633    453.0        0.66
6   train_pseudo_unlabeled      recommended   490763   4403.0        0.90
7               validation          no_idea    89431    429.0        0.48
8               validation  not_recommended    76529    447.0        0.58
9               validation      recommended   622717   4901.0        0.79
10              validation              NaN   134117    889.0        0.66
11                    test          no_idea    88994    406.0        0.46
12                    test  not_recomm